In [ ]:
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle key: ')

!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000_data

Kaggle username: rowidamohamed7
Kaggle key: ··········
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
 33% 1.72G/5.20G [01:25<03:09, 19.8MB/s]

#Check for duplicates and remove them


In [ ]:
import pandas as pd, glob

df = pd.read_csv('/content/ham10000_data/HAM10000_metadata.csv')
image_paths = glob.glob('/content/ham10000_data/**/*.jpg', recursive=True)
print(f"Metadata rows: {len(df)}")
print(f"Images found: {len(image_paths)}")

In [ ]:
!find /content/ham10000_data -name "*.jpg" | sed 's|.*/||' | sort | uniq -d | wc -l

In [ ]:
!find /content/ham10000_data -maxdepth 2 -type d

In [ ]:
!diff -rq /content/ham10000_data/HAM10000_images_part_1 /content/ham10000_data/ham10000_images_part_1 | head -5

In [ ]:
!rm -rf /content/ham10000_data/ham10000_images_part_1
!rm -rf /content/ham10000_data/ham10000_images_part_2

In [ ]:
import glob
image_paths = glob.glob('/content/ham10000_data/**/*.jpg', recursive=True)
print(f"Images found: {len(image_paths)}")

#Dataset analysis

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import glob, os, hashlib
from collections import Counter

df = pd.read_csv('/content/ham10000_data/HAM10000_metadata.csv')

Make the metadata rows point to their actual image

In [ ]:
def find_path(image_id):
    p1 = f'/content/ham10000_data/HAM10000_images_part_1/{image_id}.jpg'
    p2 = f'/content/ham10000_data/HAM10000_images_part_2/{image_id}.jpg'
    return p1 if os.path.exists(p1) else p2

df['path'] = df['image_id'].apply(find_path)


Number of images

In [ ]:
print(f"Number of images: {len(df)}")

Number of classes and their content

In [ ]:
print(f"\nClasses: {df['dx'].unique().tolist()}")
print(f"Number of classes: {df['dx'].nunique()}")

#Class explanation:


1.   bkl (Benign keratosis-like lesions): Benign
2.   nv  (Melanocytic nevi): Benign


1.   df  (Dermatofibroma): Benign
2.   mel (Melanoma): Malignant


1.   vasc (Vascular lesions): Benign
2.   bcc (Basal cell carcinoma): Malignant


1.   akiec (Actinic keratoses / intraepithelial carcinoma):precancerous









Categories precentage (Benign/Precancerous/Malignant)

In [ ]:
# Define category groups
category_map = {
    'nv': 'benign',
    'bkl': 'benign',
    'vasc': 'benign',
    'df': 'benign',
    'akiec': 'precancerous',
    'mel': 'malignant',
    'bcc': 'malignant'
}

df['category'] = df['dx'].map(category_map)

print("Counts:")
print(df['category'].value_counts())

print("\nPercentages:")
print((df['category'].value_counts(normalize=True) * 100).round(2))

Check for missing values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

Check image dimensions

In [ ]:
sample_paths = df['path'].sample(200, random_state=42)
dims = [Image.open(p).size for p in sample_paths]
dim_counts = Counter(dims)
print(f"Image dimensions (sample of 200): {dim_counts}")

All random picked 200 images are of dimension: 600x450

Check for duplicated content images

In [ ]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = df['path'].apply(file_hash)
dup_count = hashes.duplicated().sum()
print(f"Duplicate images : {dup_count}")

Remove the duplicated images

In [ ]:
df['img_hash'] = hashes
dup_mask = df['img_hash'].duplicated(keep='first')

print("Duplicate rows:")
print(df[dup_mask][['image_id', 'lesion_id', 'dx', 'path']])


df_clean = df[~dup_mask].reset_index(drop=True)
print(f"\nOriginal: {len(df)}  After removing duplicates: {len(df_clean)}")

#Data cleaning

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

Group same lesion/mole in the same test group or train group because same lesion may be photographed many times so we want to avoid data leakage

In [ ]:
lesion_df = df_clean.drop_duplicates(subset='lesion_id')[['lesion_id', 'dx']]

Split lesions into train and temp (test and validation for later)

In [ ]:
train_lesions, temp_lesions = train_test_split(
    lesion_df, test_size=0.3, stratify=lesion_df['dx'], random_state=42
)

Split temp into half test and half validation

In [ ]:
val_lesions, test_lesions = train_test_split(
    temp_lesions, test_size=0.5, stratify=temp_lesions['dx'], random_state=42
)

Map lesion splits back to full image rows

In [ ]:
train_df = df_clean[df_clean['lesion_id'].isin(train_lesions['lesion_id'])].reset_index(drop=True)
val_df   = df_clean[df_clean['lesion_id'].isin(val_lesions['lesion_id'])].reset_index(drop=True)
test_df  = df_clean[df_clean['lesion_id'].isin(test_lesions['lesion_id'])].reset_index(drop=True)

Print split sizes

In [ ]:
print(f"Train: {len(train_df)} images, {train_df['lesion_id'].nunique()} lesions")
print(f"Val:   {len(val_df)} images, {val_df['lesion_id'].nunique()} lesions")
print(f"Test:  {len(test_df)} images, {test_df['lesion_id'].nunique()} lesions")

Check for leakage

In [ ]:
assert set(train_df['lesion_id']) & set(val_df['lesion_id']) == set()
assert set(train_df['lesion_id']) & set(test_df['lesion_id']) == set()
assert set(val_df['lesion_id']) & set(test_df['lesion_id']) == set()
print("No lesion leakage across splits ")

Encode the 3 categorical labels (benign / malignant/ precancerous) (0,1,2)  instead of the 7 class labels

In [ ]:
classes = sorted(df_clean['category'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
print("Classes:", class_to_idx)

for split_df in [train_df, val_df, test_df]:
    split_df['label'] = split_df['category'].map(class_to_idx)

#Training transforms

Resize images into 224x224 to be standardized for pretrained CNNs

In [ ]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

#Validation/test transforms

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

#Custom Dataset class

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['label']
        if self.transform:
            image = self.transform(image)
        return image, label

#Instantiate datasets

In [ ]:
train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset   = HAM10000Dataset(val_df, transform=eval_transform)
test_dataset  = HAM10000Dataset(test_df, transform=eval_transform)

#Wrap in DataLoaders

In [ ]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

#Sanity check

In [ ]:
images, labels = next(iter(train_loader))
print(f"\nBatch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")
print(f"Unique labels in this batch: {labels.unique().tolist()}")

##Feature Extraction
```
Image -> CNN -> Global Average Pooling -> Embedding
```

### 1. Select baseline architecture

In [ ]:
# Supported baseline architectures and the embedding size each one produces
# once its classification head is removed (all three already end in a
# Global Average Pooling layer right before that head).
BACKBONE_CONFIGS = {
    'efficientnet_b0': {'embedding_dim': 1280},
    'resnet50':        {'embedding_dim': 2048},
    'densenet121':     {'embedding_dim': 1024},
}

BACKBONE = 'efficientnet_b0'   # <- change to 'resnet50' or 'densenet121' to switch
assert BACKBONE in BACKBONE_CONFIGS, f"Unknown backbone: {BACKBONE}"

EMBED_DIM = BACKBONE_CONFIGS[BACKBONE]['embedding_dim']
print(f"Selected baseline architecture: {BACKBONE}")
print(f"Expected embedding size: {EMBED_DIM}")

### 2. Load ImageNet pretrained weights

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_pretrained_backbone(backbone):
    """Load the chosen architecture with its ImageNet-pretrained weights."""
    if backbone == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
    elif backbone == 'resnet50':
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        model = models.resnet50(weights=weights)
    elif backbone == 'densenet121':
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        model = models.densenet121(weights=weights)
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")
    return model, weights

pretrained_model, pretrained_weights = load_pretrained_backbone(BACKBONE)
print(f"Loaded {BACKBONE} with weights: {pretrained_weights}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Build the feature extractor (CNN → Global Average Pooling → Embedding)

In [ ]:
def build_feature_extractor(model, backbone):
    """
    Truncate the classification head so the model outputs the pooled
    embedding instead of class scores:

        Image -> CNN -> Global Average Pooling -> Embedding

    Each of these torchvision models already performs global average
    pooling right before its head, so replacing that head with
    nn.Identity() is enough to expose the embedding.
    """
    if backbone == 'efficientnet_b0':
        model.classifier = nn.Identity()
    elif backbone == 'resnet50':
        model.fc = nn.Identity()
    elif backbone == 'densenet121':
        model.classifier = nn.Identity()
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    # Freeze the backbone -- we're extracting fixed features, not fine-tuning
    for param in model.parameters():
        param.requires_grad = False

    model.to(DEVICE)
    model.eval()
    return model

feature_extractor = build_feature_extractor(pretrained_model, BACKBONE)
print(f"224 x 224 x 3\n     ↓\n{BACKBONE}\n     ↓\n{EMBED_DIM}-dimensional embedding")

In [ ]:
# Sanity check: run one real batch through the extractor
images, labels = next(iter(train_loader))
images = images.to(DEVICE)

with torch.no_grad():
    embeddings = feature_extractor(images)

print(f"Input batch shape:      {images.shape}")
print(f"Output embedding shape: {embeddings.shape}")
assert embeddings.shape[1] == EMBED_DIM, "Embedding dim mismatch!"


Deep Learning / Feature Extraction


1. Build a classification model on top of EfficientNet-B0
2. Train and fine-tune it on the skin lesion dataset
3. Save the best model checkpoint
4. Extract meaningful **1280-D embeddings** for image retrieval

the Classification Model

**What we're doing:** Creating a PyTorch model that can do **two things**:
1. **Classify** a skin lesion into one of 3 categories (benign / malignant / precancerous)
2. **Extract an embedding** — a 1280-dimensional feature vector that captures the visual content

**Why both?** Training a classifier forces the model to learn features that are
*meaningful for skin lesions* (not just generic ImageNet features). After training,
we throw away the classifier and keep only the 1280-D embedding for retrieval.

**Architecture:**
```text
Image [3×224×224]
    ↓
EfficientNet-B0 conv layers (features)
    ↓
Adaptive Average Pooling → [1280]
    ↓
Dropout(0.2)
    ↓
Linear(1280 → 3) → logits
```

The `get_embedding()` method returns the 1280-D vector **before** the classifier.

In [ ]:
import copy
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

#SkinLesionModel
class SkinLesionModel(nn.Module):
    '''
    Dual-purpose model for skin lesion classification + embedding extraction.

    forward(x)        → 3 class logits   (used during training)
    get_embedding(x)  → 1280-D vector    (used for retrieval)
    '''
    def __init__(self, backbone_name='efficientnet_b0', num_classes=3, pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name
        self.num_classes = num_classes
        self.embedding_dim = BACKBONE_CONFIGS[backbone_name]['embedding_dim']  # 1280

        # Load EfficientNet-B0 with or without ImageNet weights
        if pretrained:
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
            base = models.efficientnet_b0(weights=weights)
        else:
            base = models.efficientnet_b0(weights=None)

        # Keep the convolutional backbone + global average pooling
        # but replace the original 1000-class classifier with our own
        self.features = base.features       # All convolutional layers
        self.avgpool  = base.avgpool         # AdaptiveAvgPool2d → [B, 1280, 1, 1]

        # Our new classification head (same structure as original but 3 classes)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),               # Regularisation
            nn.Linear(self.embedding_dim, num_classes)
        )

    def get_embedding(self, x):
        '''Extract the 1280-D embedding (before the classifier).'''
        x = self.features(x)       # [B, 1280, 7, 7]
        x = self.avgpool(x)        # [B, 1280, 1, 1]
        x = x.flatten(1)           # [B, 1280]
        return x

    def forward(self, x):
        '''Full forward: image → embedding → class logits.'''
        embedding = self.get_embedding(x)
        logits = self.classifier(embedding)
        return logits

print('SkinLesionModel defined ')

In [ ]:
# Sanity check: verify the model produces correct shapes
_test_model = SkinLesionModel(backbone_name=BACKBONE, num_classes=3, pretrained=True).to(DEVICE)
_test_model.eval()

_dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    _logits = _test_model(_dummy)
    _emb    = _test_model.get_embedding(_dummy)

print(f'Input shape:     {list(_dummy.shape)}')       # [2, 3, 224, 224]
print(f'Logits shape:    {list(_logits.shape)}')       # [2, 3]
print(f'Embedding shape: {list(_emb.shape)}')          # [2, 1280]
assert _logits.shape == (2, 3), 'Logits shape mismatch!'
assert _emb.shape    == (2, 1280), 'Embedding shape mismatch!'
print('\n Model produces correct shapes')
del _test_model, _dummy, _logits, _emb


Due to huge data imbalance We compute **inverse-frequency class weights** so that the loss function
penalises mistakes on rare classes more heavily. We also use **macro F1** (which treats
all classes equally) instead of accuracy as our primary metric.

We will run experiments **with and without** these weights to see the effect.

In [ ]:
# Analyse class distribution in the training set
from collections import Counter

train_counts = Counter(train_df['label'].tolist())
total_train  = sum(train_counts.values())
num_classes  = len(train_counts)

idx_to_class = {v: k for k, v in class_to_idx.items()}

print('Training set class distribution:')
print(f'{"Class":<15} {"Count":>6} {"Percentage":>10}')
print('-' * 35)
for i in range(num_classes):
    pct = train_counts[i] / total_train * 100
    print(f'{idx_to_class[i]:<15} {train_counts[i]:>6} {pct:>9.1f}%')

# Inverse-frequency weights: weight_i = total / (num_classes × count_i)
# This gives higher weight to rarer classes
class_weights = torch.tensor([
    total_train / (num_classes * train_counts[i])
    for i in range(num_classes)
], dtype=torch.float32)

print(f'\nComputed class weights: {class_weights.tolist()}')
print('  Higher weight = rarer class = penalised more when misclassified')

## Train the Baseline (Frozen Backbone)

**What we're doing:** Training the model with the EfficientNet backbone **frozen**
(no gradient updates to the conv layers). Only the new classification head learns.

**Why freeze first?**
- The pretrained ImageNet features are already very good at extracting visual patterns.
- Training the tiny classifier head is fast and stable.
- This gives us a **baseline** to compare against fine-tuning.

**Training loop anatomy:**
```text
for each epoch:
    for each batch:
        1. Forward pass   → compute predictions
        2. Loss           → measure how wrong we are
        3. Backward pass  → compute gradients
        4. Optimizer step → update weights
    Validate on held-out set
    Track best model by macro F1
```

In [ ]:
# Training and evaluation helper functions

def train_one_epoch(model, loader, criterion, optimizer, device):
    '''Train for one epoch. Returns (avg_loss, accuracy).'''
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # 1. Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 2-3. Backward pass (compute gradients)
        optimizer.zero_grad()
        loss.backward()

        # 4. Update weights
        optimizer.step()

        # Track statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    '''Evaluate the model. Returns (avg_loss, accuracy, predictions, true_labels).'''
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (running_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels))

print('Training and evaluation functions defined ')

In [ ]:
# first Frozen backbone — train only the classifier head

# Set random seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)

# Create model with fresh ImageNet weights
model = SkinLesionModel(backbone_name=BACKBONE, num_classes=3, pretrained=True)
model = model.to(DEVICE)

# Freeze the entire backbone (only classifier head will learn)
for param in model.features.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
print(f'Phase 1 — Trainable: {trainable_params:,} / {total_params:,} parameters')
print(f'  (only the classifier head is learning)\n')

# Standard CrossEntropyLoss (no class weighting yet — that's a separate experiment)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

# Training loop
NUM_EPOCHS_P1 = 10
history_p1 = {'train_loss': [], 'val_loss': [],
              'train_acc': [], 'val_acc': [], 'val_f1': []}
best_f1_p1 = 0.0
best_state_p1 = None

print('--- Phase 1: Frozen Backbone (Unweighted Loss) ---')
for epoch in range(NUM_EPOCHS_P1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, DEVICE)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    history_p1['train_loss'].append(train_loss)
    history_p1['val_loss'].append(val_loss)
    history_p1['train_acc'].append(train_acc)
    history_p1['val_acc'].append(val_acc)
    history_p1['val_f1'].append(val_f1)

    # Save the best model based on validation macro F1
    marker = ''
    if val_f1 > best_f1_p1:
        best_f1_p1 = val_f1
        best_state_p1 = copy.deepcopy(model.state_dict())
        marker = ' best'

    print(f'  Epoch {epoch+1:2d}/{NUM_EPOCHS_P1} │ '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}{marker}')

print(f'\nBest Phase 1 validation macro F1: {best_f1_p1:.4f}')

In [ ]:
# Helper functions for metrics and visualisation
os.makedirs('results/plots', exist_ok=True)

def print_detailed_metrics(y_true, y_pred, title):
    '''Print a full classification report.'''
    target_names = [idx_to_class[i] for i in range(num_classes)]
    print(f'\n{title}')
    print('=' * 60)
    print(classification_report(y_true, y_pred,
                                target_names=target_names, digits=4))


def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    '''Generate and optionally save a confusion matrix heatmap.'''
    target_names = [idx_to_class[i] for i in range(num_classes)]
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f'  Saved: {save_path}')
    plt.show()


def plot_training_curves(history, title_prefix, save_path=None):
    '''Plot loss and accuracy curves side by side.'''
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', ms=4)
    axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val',   ms=4)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title_prefix} — Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train', ms=4)
    axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val',   ms=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{title_prefix} — Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f'  Saved: {save_path}')
    plt.show()

print('Plotting helpers defined ')

In [ ]:
# Phase 1 results

# Load best Phase 1 checkpoint and evaluate
model.load_state_dict(best_state_p1)
_, _, p1_preds, p1_labels = evaluate(model, val_loader, criterion, DEVICE)

print_detailed_metrics(p1_labels, p1_preds, 'Phase 1 (Frozen Backbone) — Validation')

plot_confusion_matrix(p1_labels, p1_preds,
                      'Phase 1 — Frozen Backbone — Confusion Matrix',
                      'results/plots/confusion_matrix_phase1.png')

In [ ]:
plot_training_curves(history_p1, 'Phase 1 — Frozen Backbone',
                    'results/plots/phase1_curves.png')

Fine-Tuning ( Unfreeze Later Layers)
 Unfreezing the **last 3 blocks** of EfficientNet-B0 and
continuing training with a **smaller learning rate**.

**Why fine-tune?**
- The frozen baseline uses ImageNet features "as-is"
- Skin lesions look very different from ImageNet categories (dogs, cars, etc.)
- Fine-tuning adapts the later convolutional layers to learn **skin-lesion-specific patterns**
- Earlier layers (edges, textures) are generic and don't need updating

**Why a smaller learning rate?**
- The pretrained weights are already quite good — they encode useful visual knowledge
- A large learning rate would destroy those learned features (called **catastrophic forgetting**)
- We use **differential learning rates**: backbone layers get 1e-4, classifier gets 5e-4

```text
EfficientNet-B0 features:
  [0] [1] [2] [3] [4] [5]   ← FROZEN (generic features)
  [6] [7] [8]               ← UNFROZEN (fine-tuned for skin lesions)
  classifier                 ← UNFROZEN (already trained in Phase 1)
```

In [ ]:
# Phase 2: Unfreeze last 3 blocks and fine-tune

# Start from the best Phase 1 model
model.load_state_dict(best_state_p1)

# Unfreeze the last 3 feature blocks
for param in model.features[6:].parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 — Trainable: {trainable_params:,} / {total_params:,} parameters')
print(f'  (last 3 backbone blocks + classifier head)\n')

# Differential learning rates:
#  Backbone layers: smaller lr (1e-4) to avoid destroying pretrained features
#  Classifier head: larger lr (5e-4) to continue learning quickly
optimizer_ft = torch.optim.Adam([
    {'params': model.features[6:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(),   'lr': 5e-4},
])

# Reduce lr when validation F1 stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='max', patience=3, factor=0.5
)

NUM_EPOCHS_P2 = 10
history_p2 = {'train_loss': [], 'val_loss': [],
              'train_acc': [], 'val_acc': [], 'val_f1': []}
best_f1_p2 = best_f1_p1          # Must beat Phase 1 to count as improvement
best_state_p2 = copy.deepcopy(best_state_p1)

print('--- Phase 2: Fine-Tuning (Unweighted Loss) ---')
for epoch in range(NUM_EPOCHS_P2):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_ft, DEVICE)
    val_loss, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, DEVICE)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    scheduler.step(val_f1)

    history_p2['train_loss'].append(train_loss)
    history_p2['val_loss'].append(val_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_acc'].append(val_acc)
    history_p2['val_f1'].append(val_f1)

    marker = ''
    if val_f1 > best_f1_p2:
        best_f1_p2 = val_f1
        best_state_p2 = copy.deepcopy(model.state_dict())
        marker = '  best'

    print(f'  Epoch {epoch+1:2d}/{NUM_EPOCHS_P2} │ '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}{marker}')

print(f'\nBest Phase 2 validation macro F1: {best_f1_p2:.4f}')

In [ ]:
# Phase 2 results
model.load_state_dict(best_state_p2)
_, _, p2_preds, p2_labels = evaluate(model, val_loader, criterion, DEVICE)

print_detailed_metrics(p2_labels, p2_preds, 'Phase 2 (Fine-Tuned) — Validation')

plot_confusion_matrix(p2_labels, p2_preds,
                      'Phase 2 — Fine-Tuned — Confusion Matrix',
                      'results/plots/confusion_matrix_phase2.png')

plot_training_curves(history_p2, 'Phase 2 — Fine-Tuning',
                     'results/plots/phase2_curves.png')

In [ ]:
# ── Weighted Loss: Phase 1 (Frozen Backbone) + Phase 2 (Fine-Tuned) ──

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Fresh model
model_w = SkinLesionModel(backbone_name=BACKBONE, num_classes=3, pretrained=True)
model_w = model_w.to(DEVICE)

# Freeze backbone for Phase 1
for param in model_w.features.parameters():
    param.requires_grad = False

# Weighted loss
criterion_w = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer_w = torch.optim.Adam(model_w.classifier.parameters(), lr=1e-3)

# ── Phase 1 (weighted) ──
history_w = {'train_loss': [], 'val_loss': [],
             'train_acc': [], 'val_acc': [], 'val_f1': []}
best_f1_w = 0.0
best_state_w = None

print('--- Weighted Loss: Phase 1 (Frozen Backbone) ---')
for epoch in range(10):
    tl, ta = train_one_epoch(model_w, train_loader, criterion_w, optimizer_w, DEVICE)
    vl, va, vp, vl_true = evaluate(model_w, val_loader, criterion_w, DEVICE)
    vf = f1_score(vl_true, vp, average='macro')
    history_w['train_loss'].append(tl)
    history_w['val_loss'].append(vl)
    history_w['train_acc'].append(ta)
    history_w['val_acc'].append(va)
    history_w['val_f1'].append(vf)
    marker = ''
    if vf > best_f1_w:
        best_f1_w = vf
        best_state_w = copy.deepcopy(model_w.state_dict())
        marker = ' ◀ best'
    print(f'  Epoch {epoch+1:2d}/10 │ '
          f'Train Loss: {tl:.4f}  Acc: {ta:.4f} │ '
          f'Val Loss: {vl:.4f}  Acc: {va:.4f}  F1: {vf:.4f}{marker}')

# ── Phase 2 (weighted, fine-tuned) ──
model_w.load_state_dict(best_state_w)
for param in model_w.features[6:].parameters():
    param.requires_grad = True

optimizer_w2 = torch.optim.Adam([
    {'params': model_w.features[6:].parameters(), 'lr': 1e-4},
    {'params': model_w.classifier.parameters(),   'lr': 5e-4},
])
scheduler_w = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_w2, mode='max', patience=3, factor=0.5
)

print('\n--- Weighted Loss: Phase 2 (Fine-Tuning) ---')
for epoch in range(10):
    tl, ta = train_one_epoch(model_w, train_loader, criterion_w, optimizer_w2, DEVICE)
    vl, va, vp, vl_true = evaluate(model_w, val_loader, criterion_w, DEVICE)
    vf = f1_score(vl_true, vp, average='macro')
    scheduler_w.step(vf)
    history_w['train_loss'].append(tl)
    history_w['val_loss'].append(vl)
    history_w['train_acc'].append(ta)
    history_w['val_acc'].append(va)
    history_w['val_f1'].append(vf)
    marker = ''
    if vf > best_f1_w:
        best_f1_w = vf
        best_state_w = copy.deepcopy(model_w.state_dict())
        marker = ' ◀ best'
    print(f'  Epoch {epoch+1:2d}/10 │ '
          f'Train Loss: {tl:.4f}  Acc: {ta:.4f} │ '
          f'Val Loss: {vl:.4f}  Acc: {va:.4f}  F1: {vf:.4f}{marker}')

print(f'\nBest Weighted validation macro F1: {best_f1_w:.4f}')

In [ ]:
# ── Weighted Loss results ──
model_w.load_state_dict(best_state_w)
_, _, w_preds, w_labels = evaluate(model_w, val_loader, criterion_w, DEVICE)

print_detailed_metrics(w_labels, w_preds, 'Weighted Loss (Fine-Tuned) — Validation')

plot_confusion_matrix(w_labels, w_preds,
                      'Weighted Loss — Confusion Matrix',
                      'results/plots/confusion_matrix_weighted.png')

plot_training_curves(history_w, 'Weighted Loss (Phase 1+2)',
                     'results/plots/weighted_curves.png')

In [ ]:
# ── Comparison table ──

def compute_metrics(y_true, y_pred):
    return {
        'Accuracy':        (y_true == y_pred).mean(),
        'Macro Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Macro Recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Macro F1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted F1':     f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

comparison = pd.DataFrame({
    'Frozen Baseline':    compute_metrics(p1_labels, p1_preds),
    'Fine-Tuned':         compute_metrics(p2_labels, p2_preds),
    'Fine-Tuned+Weighted': compute_metrics(w_labels, w_preds),
}).round(4)

print('\n━━━━━━━━━━━━━━ Experiment Comparison ━━━━━━━━━━━━━━')
print(comparison.to_string())

# Save to CSV
os.makedirs('results', exist_ok=True)
comparison.to_csv('results/classification_metrics.csv')
print('\nSaved to results/classification_metrics.csv')

In [ ]:
# ── Determine and save the overall best model ──

experiments_summary = {
    'frozen_baseline':      {'f1': best_f1_p1, 'state': best_state_p1},
    'finetuned':            {'f1': best_f1_p2, 'state': best_state_p2},
    'finetuned_weighted':   {'f1': best_f1_w,  'state': best_state_w},
}

best_name = max(experiments_summary, key=lambda k: experiments_summary[k]['f1'])
best_overall_state = experiments_summary[best_name]['state']
best_overall_f1    = experiments_summary[best_name]['f1']

print(f'Best experiment: {best_name}')
print(f'Best validation macro F1: {best_overall_f1:.4f}')

# Save checkpoint
os.makedirs('models', exist_ok=True)

checkpoint = {
    'model_state_dict': best_overall_state,
    'backbone':         BACKBONE,
    'embedding_dim':    EMBED_DIM,
    'num_classes':      3,
    'class_to_idx':     class_to_idx,
    'best_experiment':  best_name,
    'best_val_macro_f1': best_overall_f1,
}

save_path = 'models/efficientnet_b0_best.pth'
torch.save(checkpoint, save_path)
print(f'\n✓ Saved best model to {save_path}')
print(f'  Backbone: {BACKBONE}')
print(f'  Embedding dim: {EMBED_DIM}')
print(f'  Classes: {class_to_idx}')

#  Embedding extraction functions

In [ ]:
# ── Embedding extraction functions ──

def extract_embedding(image_path, model, transform, device):
    '''
    Extract a 1280-D embedding from a single image.

    This is the reusable function for Member 3 and Member 4.
    Uses eval_transform (no augmentation) for deterministic results.
    '''
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)  # [1, 3, 224, 224]

    model.eval()
    with torch.no_grad():
        embedding = model.get_embedding(tensor)  # [1, 1280]

    return embedding.cpu().numpy().squeeze()  # (1280,)


@torch.no_grad()
def extract_all_embeddings(dataframe, model, transform, device, batch_size=32):
    '''
    Extract embeddings for all images in a dataframe.
    Uses eval_transform and shuffle=False to maintain alignment with metadata.

    Returns: numpy array of shape (N, 1280)
    '''
    # Create a fresh dataset with eval_transform (NO augmentation!)
    dataset = HAM10000Dataset(dataframe, transform=transform)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2)

    model.eval()
    all_embeddings = []

    for images, _ in tqdm(loader, desc='Extracting embeddings'):
        images = images.to(device)
        emb = model.get_embedding(images)  # [B, 1280]
        all_embeddings.append(emb.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)  # [N, 1280]

print('Embedding extraction functions defined ✓')

In [ ]:
#  Extract and save all embeddings
os.makedirs('embeddings', exist_ok=True)

# Load the best model for embedding extraction
best_model = SkinLesionModel(backbone_name=BACKBONE, num_classes=3, pretrained=False)
best_model.load_state_dict(best_overall_state)
best_model = best_model.to(DEVICE)
best_model.eval()

# 1. Fine-tuned embeddings (the primary embeddings for retrieval)
print('Extracting FINE-TUNED train embeddings...')
train_embeddings = extract_all_embeddings(train_df, best_model, eval_transform, DEVICE)
print(f'  Shape: {train_embeddings.shape}')   # Expected: [7001, 1280]
np.save('embeddings/train_embeddings.npy', train_embeddings)

print('Extracting FINE-TUNED test embeddings...')
test_embeddings = extract_all_embeddings(test_df, best_model, eval_transform, DEVICE)
print(f'  Shape: {test_embeddings.shape}')     # Expected: [1480, 1280]
np.save('embeddings/test_embeddings.npy', test_embeddings)

# 2. Pretrained-only embeddings (for Experiment A comparison)
print('\nExtracting PRETRAINED-ONLY embeddings (for comparison)...')
pretrained_emb_model = SkinLesionModel(
    backbone_name=BACKBONE, num_classes=3, pretrained=True)
pretrained_emb_model = pretrained_emb_model.to(DEVICE)
pretrained_emb_model.eval()

train_emb_pretrained = extract_all_embeddings(
    train_df, pretrained_emb_model, eval_transform, DEVICE)
test_emb_pretrained = extract_all_embeddings(
    test_df, pretrained_emb_model, eval_transform, DEVICE)

np.save('embeddings/train_embeddings_pretrained.npy', train_emb_pretrained)
np.save('embeddings/test_embeddings_pretrained.npy', test_emb_pretrained)
print(f'  Pretrained train: {train_emb_pretrained.shape}')
print(f'  Pretrained test:  {test_emb_pretrained.shape}')

print('\n All embeddings saved to embeddings/')

In [ ]:
#  Save metadata CSVs (row order matches embedding order)

# Training metadata
train_meta = train_df[['image_id', 'lesion_id', 'dx', 'category', 'label', 'path']].copy()
train_meta.insert(0, 'embedding_index', range(len(train_meta)))
train_meta.to_csv('embeddings/train_metadata.csv', index=False)

# Test metadata
test_meta = test_df[['image_id', 'lesion_id', 'dx', 'category', 'label', 'path']].copy()
test_meta.insert(0, 'embedding_index', range(len(test_meta)))
test_meta.to_csv('embeddings/test_metadata.csv', index=False)

# Verify alignment
assert len(train_meta) == len(train_embeddings), \
    f'MISMATCH: train_meta={len(train_meta)}, train_emb={len(train_embeddings)}'
assert len(test_meta) == len(test_embeddings), \
    f'MISMATCH: test_meta={len(test_meta)}, test_emb={len(test_embeddings)}'

print(f'Train metadata: {len(train_meta)} rows  ✓ matches {train_embeddings.shape[0]} embeddings')
print(f'Test metadata:  {len(test_meta)} rows  ✓ matches {test_embeddings.shape[0]} embeddings')
print(f'\nFirst 5 rows of train_metadata.csv:')
print(train_meta.head().to_string(index=False))

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

# L2 Normalize embeddings

def l2_normalize(embeddings):
    '''Normalize each embedding to unit length: ||x|| = 1'''
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Avoid division by zero (shouldn't happen, but safety first)
    norms = np.maximum(norms, 1e-12)
    return embeddings / norms

# Normalise fine-tuned embeddings
train_emb_norm = l2_normalize(train_embeddings)
test_emb_norm  = l2_normalize(test_embeddings)

# Normalise pretrained-only embeddings (for later comparison)
train_emb_pretrained_norm = l2_normalize(train_emb_pretrained)
test_emb_pretrained_norm  = l2_normalize(test_emb_pretrained)

# Verify: all norms should be ≈ 1.0
sample_norms = np.linalg.norm(train_emb_norm[:5], axis=1)
print(f'Sample norms after normalisation: {sample_norms}')
assert np.allclose(sample_norms, 1.0, atol=1e-6), 'Normalisation failed!'

# Save normalised embeddings
np.save('embeddings/train_embeddings_normalized.npy', train_emb_norm)
np.save('embeddings/test_embeddings_normalized.npy',  test_emb_norm)
np.save('embeddings/train_embeddings_pretrained_normalized.npy', train_emb_pretrained_norm)
np.save('embeddings/test_embeddings_pretrained_normalized.npy',  test_emb_pretrained_norm)

print(f'\nNormalised fine-tuned train:  {train_emb_norm.shape}')
print(f'Normalised fine-tuned test:   {test_emb_norm.shape}')
print(f'Normalised pretrained train:  {train_emb_pretrained_norm.shape}')
print(f'Normalised pretrained test:   {test_emb_pretrained_norm.shape}')
print('\n✓ All normalised embeddings saved')

In [ ]:
# Build FAISS Inner Product Index

dim = train_emb_norm.shape[1]  # 1280
index = faiss.IndexFlatIP(dim)

# Add normalised training embeddings (these form the retrieval database)
index.add(train_emb_norm.astype(np.float32))

# Sanity checks
print(f'FAISS index dimension:    {dim}')
print(f'Vectors in FAISS index:   {index.ntotal}')
print(f'Train embeddings:         {len(train_emb_norm)}')
print(f'Train metadata rows:      {len(train_meta)}')

assert index.ntotal == len(train_emb_norm), 'FAISS count ≠ embeddings count!'
assert index.ntotal == len(train_meta),     'FAISS count ≠ metadata count!'
print('\n All counts match — index is correctly built')

# Save the index
faiss.write_index(index, 'embeddings/faiss_index.bin')
print(f'\nSaved FAISS index to embeddings/faiss_index.bin')

# Verify it loads correctly
index_check = faiss.read_index('embeddings/faiss_index.bin')
assert index_check.ntotal == index.ntotal, 'Loaded index has wrong count!'
print(f'Verified: reloaded index has {index_check.ntotal} vectors')

In [ ]:
# Sanity check: query the first test image

K = 5
query_idx = 0
query_embedding = test_emb_norm[query_idx:query_idx+1].astype(np.float32)

query_info = test_meta.iloc[query_idx]
print(f'Query image:   {query_info["image_id"]}')
print(f'Query label:   {query_info["category"]}')
print(f'Query lesion:  {query_info["lesion_id"]}')

# Search!
similarities, indices = index.search(query_embedding, K)

print(f'\nTop-{K} Retrieved Images:')
print(f'{"Rank":<6} {"Image ID":<16} {"Category":<15} {"Similarity":<10}')
print('─' * 50)
for rank, (sim, idx) in enumerate(zip(similarities[0], indices[0]), 1):
    retrieved = train_meta.iloc[idx]
    match = '' if retrieved['category'] == query_info['category'] else '✗'
    print(f'{rank:<6} {retrieved["image_id"]:<16} '
          f'{retrieved["category"]:<15} {sim:<10.4f} {match}')

# Verify: query must NOT be in the results
retrieved_ids = set(train_meta.iloc[indices[0]]['image_id'])
assert query_info['image_id'] not in retrieved_ids, \
    'ERROR: Query found in results! Test/train separation violated.'
print('\n Query is NOT in the retrieved results (correct separation)')
print('Retrieval pipeline is working end-to-end')

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — retrieve_similar_images()
# Wraps the pipeline (image → embedding → normalize → FAISS search)
# into one reusable function for evaluation and for the app.
# ══════════════════════════════════════════════════════

def retrieve_similar_images(image_path, model, transform, index, metadata_df,
                             device=DEVICE, k=5):
    '''
    Given a path to a skin lesion image, returns the top-k most similar
    images from the retrieval database (the FAISS index), along with
    their metadata and similarity scores.

    Args:
        image_path:   path to the query image (str)
        model:        trained SkinLesionModel (must have get_embedding())
        transform:    eval_transform (no augmentation — deterministic)
        index:        FAISS index built from the training embeddings
        metadata_df:  metadata dataframe aligned with the FAISS index
                      (e.g. train_meta — row i in metadata_df must match
                      vector i in the index)
        device:       torch device
        k:            number of neighbors to retrieve

    Returns:
        pandas DataFrame with the top-k rows from metadata_df, plus a
        'similarity' column, sorted by similarity (highest first).
    '''
    # 1. Image -> 1280-D embedding (reuses Member 2's function)
    embedding = extract_embedding(image_path, model, transform, device)

    # 2. L2-normalize so inner product == cosine similarity
    embedding = l2_normalize(embedding.reshape(1, -1)).astype(np.float32)

    # 3. Search the FAISS index
    similarities, indices = index.search(embedding, k)

    # 4. Attach metadata + similarity scores to the results
    results = metadata_df.iloc[indices[0]].copy().reset_index(drop=True)
    results['similarity'] = similarities[0]

    return results


# ── Quick test: run it on a real test-set image ──
sample_query_path = test_meta.iloc[0]['path']
sample_query_label = test_meta.iloc[0]['category']

print(f'Query image:  {test_meta.iloc[0]["image_id"]}')
print(f'Query label:  {sample_query_label}\n')

results = retrieve_similar_images(
    image_path=sample_query_path,
    model=best_model,
    transform=eval_transform,
    index=index,
    metadata_df=train_meta,
    k=5
)

print('Top-5 Retrieved Images:')
results[['image_id', 'category', 'similarity']]

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — Step 2: Run retrieval over the ENTIRE test set
# Uses the already-extracted, already-normalized test embeddings
# (test_emb_norm) and searches the FAISS index in one batched call
# instead of looping image-by-image (much faster).
# ══════════════════════════════════════════════════════

K_EVAL = 10  # retrieve top-10 so we can compute Precision@1, @5 and Recall@5, @10 from one pass

# Batched search: query with ALL test embeddings at once
all_similarities, all_indices = index.search(test_emb_norm.astype(np.float32), K_EVAL)

print(f'Queried {all_similarities.shape[0]} test images')
print(f'Retrieved top-{K_EVAL} neighbors for each\n')

# Build one row per test query, with its true label and the labels/
# similarities of its top-10 retrieved neighbors from the train set
eval_records = []

for i in range(len(test_meta)):
    query_label = test_meta.iloc[i]['category']
    query_image_id = test_meta.iloc[i]['image_id']

    neighbor_positions = all_indices[i]          # row indices into train_meta
    neighbor_sims = all_similarities[i]           # cosine similarities

    neighbor_labels = train_meta.iloc[neighbor_positions]['category'].values
    neighbor_image_ids = train_meta.iloc[neighbor_positions]['image_id'].values

    eval_records.append({
        'query_image_id': query_image_id,
        'query_label': query_label,
        'retrieved_image_ids': neighbor_image_ids,
        'retrieved_labels': neighbor_labels,
        'similarities': neighbor_sims
    })

print('Example record (first test query):')
print(f"  Query:     {eval_records[0]['query_image_id']}  (label: {eval_records[0]['query_label']})")
print(f"  Retrieved: {list(eval_records[0]['retrieved_image_ids'])}")
print(f"  Labels:    {list(eval_records[0]['retrieved_labels'])}")
print(f"  Sims:      {[round(s, 3) for s in eval_records[0]['similarities']]}")

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — Step 3: Precision@1, Precision@5, Recall@5, Recall@10, mAP
# Computed from eval_records built in Step 2.
# ══════════════════════════════════════════════════════

# Recall needs to know: for each class, how many total same-class
# items exist in the retrieval database (train set)? That's the
# denominator — "how many relevant items COULD we have found".
train_class_counts = train_meta['category'].value_counts().to_dict()
print('Train set size per class (retrieval database):')
print(train_class_counts, '\n')


def precision_at_k(retrieved_labels, query_label, k):
    '''Fraction of the top-k retrieved items that share the query's label.'''
    top_k = retrieved_labels[:k]
    hits = np.sum(top_k == query_label)
    return hits / k


def recall_at_k(retrieved_labels, query_label, k, total_relevant):
    '''Fraction of ALL same-label items in the database found in the top-k.'''
    top_k = retrieved_labels[:k]
    hits = np.sum(top_k == query_label)
    return hits / total_relevant if total_relevant > 0 else 0.0


def average_precision(retrieved_labels, query_label):
    '''
    Average Precision for one query: precision computed at each
    position where a correct (same-label) item appears, averaged
    over the number of correct items found in the retrieved list.
    '''
    relevant_flags = (retrieved_labels == query_label).astype(int)
    if relevant_flags.sum() == 0:
        return 0.0

    cumulative_hits = np.cumsum(relevant_flags)
    precisions_at_hits = cumulative_hits / (np.arange(len(relevant_flags)) + 1)
    # Only count precision at positions where a hit actually occurred
    ap = np.sum(precisions_at_hits * relevant_flags) / relevant_flags.sum()
    return ap


# ── Compute metrics for every query, then average ──
p_at_1_list, p_at_5_list = [], []
r_at_5_list, r_at_10_list = [], []
ap_list = []

for record in eval_records:
    query_label = record['query_label']
    retrieved_labels = record['retrieved_labels']
    total_relevant = train_class_counts.get(query_label, 0)

    p_at_1_list.append(precision_at_k(retrieved_labels, query_label, k=1))
    p_at_5_list.append(precision_at_k(retrieved_labels, query_label, k=5))
    r_at_5_list.append(recall_at_k(retrieved_labels, query_label, k=5, total_relevant=total_relevant))
    r_at_10_list.append(recall_at_k(retrieved_labels, query_label, k=10, total_relevant=total_relevant))
    ap_list.append(average_precision(retrieved_labels, query_label))

overall_metrics = {
    'Precision@1': np.mean(p_at_1_list),
    'Precision@5': np.mean(p_at_5_list),
    'Recall@5':    np.mean(r_at_5_list),
    'Recall@10':   np.mean(r_at_10_list),
    'mAP':         np.mean(ap_list),
}

print('═' * 45)
print('Overall Retrieval Metrics (Fine-Tuned Embeddings)')
print('═' * 45)
for metric_name, value in overall_metrics.items():
    print(f'{metric_name:<12}: {value:.4f}')

# Save to results/ for the final report
os.makedirs('results', exist_ok=True)
pd.DataFrame([overall_metrics]).to_csv('results/metrics.csv', index=False)
print('\nSaved to results/metrics.csv')

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — Step 4: Per-class metrics + confusion matrix
# Breaks down Precision@k / Recall@k by category, since the dataset
# is imbalanced (mostly benign) — the malignant/precancerous numbers
# are the ones that actually matter clinically.
# ══════════════════════════════════════════════════════

from collections import defaultdict

# ── Per-class Precision@k / Recall@k ──
per_class_p1   = defaultdict(list)
per_class_p5   = defaultdict(list)
per_class_r5   = defaultdict(list)
per_class_r10  = defaultdict(list)
per_class_ap   = defaultdict(list)

for record in eval_records:
    query_label = record['query_label']
    retrieved_labels = record['retrieved_labels']
    total_relevant = train_class_counts.get(query_label, 0)

    per_class_p1[query_label].append(precision_at_k(retrieved_labels, query_label, k=1))
    per_class_p5[query_label].append(precision_at_k(retrieved_labels, query_label, k=5))
    per_class_r5[query_label].append(recall_at_k(retrieved_labels, query_label, k=5, total_relevant=total_relevant))
    per_class_r10[query_label].append(recall_at_k(retrieved_labels, query_label, k=10, total_relevant=total_relevant))
    per_class_ap[query_label].append(average_precision(retrieved_labels, query_label))

categories = sorted(per_class_p1.keys())

per_class_table = pd.DataFrame({
    'Category':     categories,
    'Test Count':   [len(per_class_p1[c]) for c in categories],
    'Precision@1':  [np.mean(per_class_p1[c])  for c in categories],
    'Precision@5':  [np.mean(per_class_p5[c])  for c in categories],
    'Recall@5':     [np.mean(per_class_r5[c])  for c in categories],
    'Recall@10':    [np.mean(per_class_r10[c]) for c in categories],
    'mAP':          [np.mean(per_class_ap[c])  for c in categories],
}).round(4)

print('═' * 70)
print('Per-Class Retrieval Metrics')
print('═' * 70)
print(per_class_table.to_string(index=False))

os.makedirs('results', exist_ok=True)
per_class_table.to_csv('results/per_class_metrics.csv', index=False)


# ── Retrieval-as-classifier confusion matrix ──
# Assign each query the MAJORITY label among its top-5 neighbors,
# then compare that predicted label against the true label.
def majority_vote_label(retrieved_labels, k=5):
    top_k = retrieved_labels[:k]
    vals, counts = np.unique(top_k, return_counts=True)
    return vals[np.argmax(counts)]

true_labels_all = [r['query_label'] for r in eval_records]
predicted_labels_all = [majority_vote_label(r['retrieved_labels'], k=5) for r in eval_records]

print('\n' + '═' * 45)
print('Retrieval-as-Classifier — Classification Report')
print('═' * 45)
print(classification_report(true_labels_all, predicted_labels_all, digits=4))

cm = confusion_matrix(true_labels_all, predicted_labels_all, labels=categories)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories, ax=ax)
ax.set_xlabel('Predicted (majority vote of top-5 retrieved)')
ax.set_ylabel('True label')
ax.set_title('Retrieval-Based Classification — Confusion Matrix')
plt.tight_layout()

os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/retrieval_confusion_matrix.png', dpi=150)
plt.show()

print('\nSaved: results/per_class_metrics.csv, results/plots/retrieval_confusion_matrix.png')

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — Step 5: Compare Pretrained vs Fine-Tuned embeddings
# The core research experiment: does fine-tuning on skin lesions
# actually produce a BETTER retrieval space than generic ImageNet
# features? We rerun the exact same evaluation pipeline (Steps 2-4)
# on the pretrained-only embeddings and put both side by side.
# ══════════════════════════════════════════════════════

def evaluate_retrieval_system(train_emb_norm_, test_emb_norm_, train_meta_, test_meta_, k_eval=10):
    '''
    Builds a FAISS index from train_emb_norm_, searches it with every
    embedding in test_emb_norm_, and returns overall + per-class metrics.
    This is Steps 2-4 wrapped into one reusable function so we can run
    the identical evaluation on any set of embeddings.
    '''
    # Build a fresh index for this embedding set
    dim = train_emb_norm_.shape[1]
    local_index = faiss.IndexFlatIP(dim)
    local_index.add(train_emb_norm_.astype(np.float32))

    # Batched search (Step 2)
    sims, idxs = local_index.search(test_emb_norm_.astype(np.float32), k_eval)

    train_class_counts_ = train_meta_['category'].value_counts().to_dict()

    p1_list, p5_list, r5_list, r10_list, ap_list = [], [], [], [], []
    per_class = defaultdict(lambda: defaultdict(list))

    for i in range(len(test_meta_)):
        query_label = test_meta_.iloc[i]['category']
        retrieved_labels = train_meta_.iloc[idxs[i]]['category'].values
        total_relevant = train_class_counts_.get(query_label, 0)

        p1 = precision_at_k(retrieved_labels, query_label, k=1)
        p5 = precision_at_k(retrieved_labels, query_label, k=5)
        r5 = recall_at_k(retrieved_labels, query_label, k=5, total_relevant=total_relevant)
        r10 = recall_at_k(retrieved_labels, query_label, k=10, total_relevant=total_relevant)
        ap = average_precision(retrieved_labels, query_label)

        p1_list.append(p1); p5_list.append(p5)
        r5_list.append(r5); r10_list.append(r10)
        ap_list.append(ap)

        per_class['Precision@1'][query_label].append(p1)
        per_class['Precision@5'][query_label].append(p5)
        per_class['Recall@5'][query_label].append(r5)
        per_class['Recall@10'][query_label].append(r10)
        per_class['mAP'][query_label].append(ap)

    overall = {
        'Precision@1': np.mean(p1_list),
        'Precision@5': np.mean(p5_list),
        'Recall@5':    np.mean(r5_list),
        'Recall@10':   np.mean(r10_list),
        'mAP':         np.mean(ap_list),
    }

    per_class_table_ = pd.DataFrame({
        metric: {cat: np.mean(vals) for cat, vals in per_class[metric].items()}
        for metric in per_class
    }).round(4)

    return overall, per_class_table_


# ── Run on FINE-TUNED embeddings (reproduces Step 3's numbers, as a check) ──
print('Evaluating FINE-TUNED embeddings...')
finetuned_overall, finetuned_per_class = evaluate_retrieval_system(
    train_emb_norm, test_emb_norm, train_meta, test_meta
)

# ── Run on PRETRAINED-ONLY embeddings ──
print('Evaluating PRETRAINED-ONLY embeddings...')
pretrained_overall, pretrained_per_class = evaluate_retrieval_system(
    train_emb_pretrained_norm, test_emb_pretrained_norm, train_meta, test_meta
)

# ── Side-by-side comparison ──
comparison_table = pd.DataFrame({
    'Pretrained (ImageNet)': pretrained_overall,
    'Fine-Tuned (Ours)':     finetuned_overall,
}).round(4)

comparison_table['Improvement'] = (
    comparison_table['Fine-Tuned (Ours)'] - comparison_table['Pretrained (ImageNet)']
).round(4)

print('\n' + '═' * 55)
print('Retrieval Comparison: Pretrained vs Fine-Tuned')
print('═' * 55)
print(comparison_table.to_string())

os.makedirs('results', exist_ok=True)
comparison_table.to_csv('results/pretrained_vs_finetuned_comparison.csv')
print('\nSaved to results/pretrained_vs_finetuned_comparison.csv')

In [ ]:
# ══════════════════════════════════════════════════════
# Member 3 — Step 6: Similarity distribution plots
# Shows whether the embedding space actually separates classes:
# similarity scores for CORRECT matches (same category) should be
# noticeably higher than for INCORRECT matches (different category).
# Uses eval_records from Step 2 (fine-tuned embeddings, top-10 each).
# ══════════════════════════════════════════════════════

correct_sims = []
incorrect_sims = []

for record in eval_records:
    query_label = record['query_label']
    retrieved_labels = record['retrieved_labels']
    sims = record['similarities']

    for label, sim in zip(retrieved_labels, sims):
        if label == query_label:
            correct_sims.append(sim)
        else:
            incorrect_sims.append(sim)

correct_sims = np.array(correct_sims)
incorrect_sims = np.array(incorrect_sims)

print(f'Correct matches:   {len(correct_sims)}  (mean similarity: {correct_sims.mean():.4f})')
print(f'Incorrect matches: {len(incorrect_sims)}  (mean similarity: {incorrect_sims.mean():.4f})')
print(f'Separation (correct - incorrect): {correct_sims.mean() - incorrect_sims.mean():.4f}')

# ── Overlaid histogram: correct vs incorrect ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlaid histograms
axes[0].hist(correct_sims, bins=40, alpha=0.6, label='Correct (same category)', color='green', density=True)
axes[0].hist(incorrect_sims, bins=40, alpha=0.6, label='Incorrect (different category)', color='red', density=True)
axes[0].axvline(correct_sims.mean(), color='green', linestyle='--', linewidth=1)
axes[0].axvline(incorrect_sims.mean(), color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('Similarity Distribution: Correct vs Incorrect Retrievals')
axes[0].legend()

# Right: same thing as boxplots, easier to compare spread at a glance
axes[1].boxplot([correct_sims, incorrect_sims], labels=['Correct', 'Incorrect'])
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Similarity Spread by Match Correctness')

plt.tight_layout()

os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/similarity_distribution.png', dpi=150)
plt.show()

print('\nSaved: results/plots/similarity_distribution.png')


# ── Bonus: per-class version — is separation better for some classes? ──
fig, ax = plt.subplots(figsize=(9, 5))

for category in sorted(set(r['query_label'] for r in eval_records)):
    cat_correct = []
    for record in eval_records:
        if record['query_label'] != category:
            continue
        for label, sim in zip(record['retrieved_labels'], record['similarities']):
            if label == category:
                cat_correct.append(sim)
    if cat_correct:
        sns.kdeplot(cat_correct, label=f'{category} (correct matches)', ax=ax)

ax.set_xlabel('Cosine Similarity')
ax.set_title('Correct-Match Similarity by Category')
ax.legend()
plt.tight_layout()
plt.savefig('results/plots/similarity_distribution_per_class.png', dpi=150)
plt.show()

print('Saved: results/plots/similarity_distribution_per_class.png')

In [ ]:
#import faiss
from retrieval import retrieve_similar_images
from evaluation import evaluate_retrieval_system, precision_at_k

print("Imports work — files are valid standalone modules.")

In [ ]:
!python build_index.py

In [ ]:
import os
print(os.getcwd())
print(os.listdir('.'))

In [ ]:
%%writefile -a evaluation.py

import matplotlib.pyplot as plt
from PIL import Image
import os


def save_retrieval_example(query_row, retrieved_df, save_path, k=5):
    """
    Saves a figure showing the query image next to its top-k retrieved
    matches, each labeled with category + similarity score.
    """
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3.5))

    # Query image (leftmost)
    query_img = Image.open(query_row['path']).convert('RGB')
    axes[0].imshow(query_img)
    axes[0].set_title(f"QUERY\n{query_row['category']}", fontweight='bold', color='blue')
    axes[0].axis('off')

    # Retrieved images
    for i in range(k):
        row = retrieved_df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        axes[i + 1].imshow(img)

        is_correct = row['category'] == query_row['category']
        color = 'green' if is_correct else 'red'
        axes[i + 1].set_title(
            f"#{i+1}: {row['category']}\nsim={row['similarity']:.3f}",
            color=color, fontsize=10
        )
        axes[i + 1].axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os


def save_retrieval_example(query_row, retrieved_df, save_path, k=5):
    """
    Saves a figure showing the query image next to its top-k retrieved
    matches, each labeled with category + similarity score.
    """
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3.5))

    query_img = Image.open(query_row['path']).convert('RGB')
    axes[0].imshow(query_img)
    axes[0].set_title(f"QUERY\n{query_row['category']}", fontweight='bold', color='blue')
    axes[0].axis('off')

    for i in range(k):
        row = retrieved_df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        axes[i + 1].imshow(img)

        is_correct = row['category'] == query_row['category']
        color = 'green' if is_correct else 'red'
        axes[i + 1].set_title(
            f"#{i+1}: {row['category']}\nsim={row['similarity']:.3f}",
            color=color, fontsize=10
        )
        axes[i + 1].axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

In [ ]:
os.makedirs('results/retrieval_examples', exist_ok=True)

example_categories = test_meta['category'].unique()

for category in example_categories:
    query_row = test_meta[test_meta['category'] == category].iloc[0]

    results = retrieve_similar_images(
        image_path=query_row['path'],
        model=best_model,
        transform=eval_transform,
        index=index,
        metadata_df=train_meta,
        device=DEVICE,          # ← add this line
        k=5
    )

    save_path = f'results/retrieval_examples/example_{category}.png'
    save_retrieval_example(query_row, results, save_path, k=5)
    print(f'Saved: {save_path}')